# GraphRAG Finserv Compliance Agent — Prototype

This notebook is a **thin orchestration layer**. All business logic lives in `src/`.

## What this notebook covers
1. Setup & environment loading
2. Neo4j connection test
3. Run a sample Cypher query helper
4. GraphRAG retriever with a sample question
5. Full compliance agent with an APRA/AML query

---
> **Prerequisites:** `pip install -r requirements.txt` and a populated `.env` file.

## 1. Setup & Environment Loading

In [ ]:
import sys
import os

# Add the project root to the Python path so src/ is importable
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv

# Load .env from project root
env_path = os.path.join(project_root, ".env")
loaded = load_dotenv(env_path)

print(f"✓ .env loaded: {loaded}")
print(f"  NEO4J_URI      : {'set' if os.getenv('NEO4J_URI') else 'MISSING'}")
print(f"  NEO4J_USERNAME : {'set' if os.getenv('NEO4J_USERNAME') else 'MISSING'}")
print(f"  NEO4J_PASSWORD : {'set' if os.getenv('NEO4J_PASSWORD') else 'MISSING'}")
print(f"  ANTHROPIC_API_KEY : {'set' if os.getenv('ANTHROPIC_API_KEY') else 'MISSING'}")

## 2. Neo4j Connection Test

In [ ]:
from src.graph.connection import Neo4jConnection

conn = Neo4jConnection()
conn.connect()

# Simple connectivity check
result = conn.run_query("RETURN 'Connection OK' AS status")
print(result)

## 3. Sample Cypher Query Helper

Run a query from `src/graph/queries.py` to verify the graph has data.

In [ ]:
from src.graph.queries import get_loan_accounts, get_compliance_flags
import json

# --- Entity layer: loan accounts ---
accounts = get_loan_accounts(conn, limit=5)
print(f"Loan accounts found: {len(accounts)}")
for acct in accounts:
    print(" ", acct)

print()

# --- Runtime assessment layer: open compliance flags ---
flags = get_compliance_flags(conn)
print(f"Open compliance flags: {len(flags)}")
for flag in flags[:5]:
    print(" ", flag)

## 4. GraphRAG Retriever

Convert a natural language question to Cypher via Claude, run it on Neo4j, and format the results.

In [ ]:
from src.retriever.graphrag import GraphRAGRetriever

retriever = GraphRAGRetriever(neo4j_conn=conn)

sample_question = "Which loan accounts have a high risk rating and are currently under review?"

print("Question:", sample_question)
print("-" * 60)

# Step 1: NL → Cypher
cypher = retriever.nl_to_cypher(sample_question)
print("Generated Cypher:")
print(cypher)
print()

# Step 2: Execute + format
context = retriever.retrieve_and_format(sample_question)
print("Formatted Context:")
print(context)

## 5. Full Compliance Agent

Run the agentic loop with an APRA-focused compliance query. Claude will autonomously decide which graph tools to call and produce a structured compliance finding.

In [ ]:
from src.agent.compliance_agent import ComplianceAgent

agent = ComplianceAgent(neo4j_conn=conn)

compliance_query = (
    "Are there any loan accounts showing suspicious transaction patterns "
    "that may breach APRA prudential standards?"
)

print("Query:", compliance_query)
print("=" * 70)
print()

response = agent.run(compliance_query)

print("Agent Response:")
print("-" * 70)
print(response)

## 6. Cleanup

In [ ]:
conn.close()
print("Neo4j connection closed.")

---

## Next Steps

- [ ] Seed the Neo4j AuraDB with `data/synthetic/loans.json` and `data/synthetic/regulations.json`
- [ ] Refine the `SYSTEM_PROMPT` in `src/agent/compliance_agent.py` with organisation-specific context
- [ ] Add more tools in `src/agent/tools.py` as the graph schema expands
- [ ] Build a `02_data_loader.ipynb` to ingest and validate graph data
- [ ] Add structured output (Pydantic) for agent compliance reports